# Garmin Self-Experimentation LabExploratory data analysis of daily physiological metrics from Garmin watch data.**Data range:** ~16 days (Feb 5–21, 2026) — early-stage collection.

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.dates as mdatesimport seaborn as snsfrom datetime import datetimesns.set_theme(style="whitegrid", palette="muted")plt.rcParams["figure.figsize"] = (14, 5)plt.rcParams["figure.dpi"] = 100# Load dataclean = pd.read_csv("data/clean/daily_hybrid.csv", parse_dates=["date"])feats = pd.read_csv("data/model/daily_features.csv", parse_dates=["date"])acts = pd.read_csv("data/activities.csv")acts["date"] = pd.to_datetime(acts["startTimeLocal"]).dt.dateprint(f"Clean daily data: {len(clean)} rows, {len(clean.columns)} columns")print(f"Feature matrix: {len(feats)} rows, {len(feats.columns)} columns")print(f"Activities: {len(acts)} rows")print(f"Date range: {clean['date'].min().date()} to {clean['date'].max().date()}")

## 1. Data Completeness

In [ ]:
# Show non-null counts for key columnskey_cols = [    "sleep_total_hrs", "sleep_score", "sleep_deep_pct", "sleep_rem_pct",    "resting_hr", "hrv_last_night", "stress_avg", "avg_spo2_sleep",    "avg_respiration_sleep", "body_battery_charged", "body_battery_drained",    "training_readiness_score", "tr_acute_load", "vo2max",    "bed_time_hour", "wake_time_hour",]available = [c for c in key_cols if c in clean.columns]completeness = clean[available].notna().sum() / len(clean) * 100print("Data completeness (% non-null):")print(completeness.sort_values(ascending=False).round(1).to_string())

In [ ]:
# Visual completeness heatmapfig, ax = plt.subplots(figsize=(16, 6))avail_numeric = [c for c in available if clean[c].dtype != "object"]data_matrix = clean[avail_numeric].notna().astype(int).Tsns.heatmap(data_matrix, cmap="YlGn", cbar_kws={"label": "Has data"},            xticklabels=clean["date"].dt.strftime("%m-%d"), yticklabels=avail_numeric, ax=ax)ax.set_title("Data Availability by Date")ax.set_xlabel("Date")plt.tight_layout()plt.show()

## 2. Descriptive Statistics

In [ ]:
# Core metrics summarycore = ["sleep_total_hrs", "sleep_score", "sleep_deep_pct", "sleep_rem_pct",        "resting_hr", "hrv_last_night", "stress_avg", "body_battery_charged",        "training_readiness_score", "bed_time_hour", "wake_time_hour"]core_avail = [c for c in core if c in clean.columns]print(clean[core_avail].describe().round(2).to_string())

## 3. Time Series — Key Metrics

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)# Sleepax = axes[0]if "sleep_total_hrs" in clean.columns:    ax.bar(clean["date"], clean["sleep_total_hrs"], alpha=0.6, label="Sleep (hrs)", color="steelblue")    if "sleep_total_hrs_7d_avg" in clean.columns:        ax.plot(clean["date"], clean["sleep_total_hrs_7d_avg"], "r-", lw=2, label="7d avg")    ax.axhline(7, ls="--", color="green", alpha=0.5, label="7h target")    ax.axhline(6, ls="--", color="red", alpha=0.5, label="6h minimum")ax.set_ylabel("Hours")ax.set_title("Sleep Duration")ax.legend(loc="upper left")# HRV + Resting HR (dual axis)ax = axes[1]if "hrv_last_night" in clean.columns:    ax.plot(clean["date"], clean["hrv_last_night"], "o-", color="teal", label="HRV (ms)", markersize=5)ax.set_ylabel("HRV (ms)", color="teal")ax2 = ax.twinx()if "resting_hr" in clean.columns:    ax2.plot(clean["date"], clean["resting_hr"], "s-", color="salmon", label="Resting HR", markersize=5)ax2.set_ylabel("Resting HR (bpm)", color="salmon")ax.set_title("HRV & Resting Heart Rate")lines1, labels1 = ax.get_legend_handles_labels()lines2, labels2 = ax2.get_legend_handles_labels()ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")# Stress + Body Batteryax = axes[2]if "stress_avg" in clean.columns:    ax.plot(clean["date"], clean["stress_avg"], "o-", color="coral", label="Avg Stress", markersize=5)if "body_battery_charged" in clean.columns:    ax2 = ax.twinx()    ax2.bar(clean["date"], clean["body_battery_charged"], alpha=0.3, color="green", label="BB Charged")    ax2.bar(clean["date"], -clean["body_battery_drained"], alpha=0.3, color="red", label="BB Drained")    ax2.set_ylabel("Body Battery")ax.set_ylabel("Stress Level")ax.set_title("Stress & Body Battery")lines1, labels1 = ax.get_legend_handles_labels()if "body_battery_charged" in clean.columns:    lines2, labels2 = ax2.get_legend_handles_labels()    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")# Training Readinessax = axes[3]if "training_readiness_score" in clean.columns:    colors = clean["training_readiness_score"].apply(        lambda x: "red" if pd.notna(x) and x < 50 else ("green" if pd.notna(x) and x >= 75 else "orange"))    ax.bar(clean["date"], clean["training_readiness_score"], color=colors, alpha=0.7)    ax.axhline(50, ls="--", color="red", alpha=0.5)    ax.axhline(75, ls="--", color="green", alpha=0.5)ax.set_ylabel("Score")ax.set_title("Training Readiness")for ax in axes:    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))plt.tight_layout()plt.show()

## 4. Sleep Architecture

In [ ]:
if all(c in clean.columns for c in ["sleep_deep_hrs", "sleep_rem_hrs", "sleep_light_hrs", "sleep_awake_hrs"]):    fig, axes = plt.subplots(1, 2, figsize=(14, 5))    # Stacked bar    ax = axes[0]    dates = clean["date"]    bottom = np.zeros(len(clean))    for stage, color, label in [        ("sleep_deep_hrs", "#1a237e", "Deep"),        ("sleep_rem_hrs", "#4a148c", "REM"),        ("sleep_light_hrs", "#7986cb", "Light"),        ("sleep_awake_hrs", "#ef5350", "Awake"),    ]:        vals = clean[stage].fillna(0).values        ax.bar(dates, vals, bottom=bottom, color=color, label=label, alpha=0.8)        bottom += vals    ax.set_ylabel("Hours")    ax.set_title("Sleep Architecture (Stacked)")    ax.legend()    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))    ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))    # Percentages over time    ax = axes[1]    for col, label, color in [        ("sleep_deep_pct", "Deep %", "#1a237e"),        ("sleep_rem_pct", "REM %", "#4a148c"),        ("sleep_light_pct", "Light %", "#7986cb"),    ]:        if col in clean.columns:            ax.plot(dates, clean[col], "o-", label=label, color=color, markersize=5)    ax.set_ylabel("Percentage")    ax.set_title("Sleep Stage Proportions")    ax.legend()    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))    ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))    plt.tight_layout()    plt.show()

## 5. Sleep Timing

In [ ]:
if "bed_time_hour" in clean.columns and "wake_time_hour" in clean.columns:    fig, ax = plt.subplots(figsize=(14, 5))    # Adjust bed_time for plotting (values > 12 are PM, < 12 are AM next day)    bed = clean["bed_time_hour"].copy()    wake = clean["wake_time_hour"].copy()    ax.plot(clean["date"], bed, "o-", color="indigo", label="Bedtime (hour)", markersize=6)    ax.plot(clean["date"], wake, "s-", color="orange", label="Wake time (hour)", markersize=6)    ax.set_ylabel("Hour of day")    ax.set_title("Sleep Timing")    ax.legend()    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))    plt.tight_layout()    plt.show()    print(f"Average bedtime: {bed.mean():.1f}h ({int(bed.mean())}:{int((bed.mean()%1)*60):02d})")    print(f"Average wake time: {wake.mean():.1f}h ({int(wake.mean())}:{int((wake.mean()%1)*60):02d})")

## 6. Correlation Matrix

In [ ]:
corr_cols = [    "sleep_total_hrs", "sleep_score", "sleep_deep_pct", "sleep_rem_pct",    "resting_hr", "hrv_last_night", "stress_avg",    "body_battery_charged", "body_battery_drained",    "training_readiness_score", "avg_sleep_stress",    "bed_time_hour", "avg_spo2_sleep",]corr_avail = [c for c in corr_cols if c in clean.columns]corr = clean[corr_avail].corr()fig, ax = plt.subplots(figsize=(12, 10))mask = np.triu(np.ones_like(corr, dtype=bool))sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,            vmin=-1, vmax=1, square=True, ax=ax)ax.set_title("Correlation Matrix — Daily Metrics")plt.tight_layout()plt.show()# Top correlationspairs = []for i in range(len(corr_avail)):    for j in range(i+1, len(corr_avail)):        pairs.append((corr_avail[i], corr_avail[j], corr.iloc[i,j]))pairs.sort(key=lambda x: abs(x[2]), reverse=True)print("Top 10 correlations (by magnitude):")for a, b, r in pairs[:10]:    print(f"  {a:30s} <-> {b:30s}  r = {r:+.3f}")

## 7. Activities Overview

In [ ]:
acts_df = pd.read_csv("data/activities.csv")acts_df["date"] = pd.to_datetime(acts_df["startTimeLocal"]).dt.dateacts_df["distance_km"] = pd.to_numeric(acts_df.get("distance", pd.Series(dtype=float)), errors="coerce") / 1000acts_df["duration_min"] = pd.to_numeric(acts_df.get("duration", pd.Series(dtype=float)), errors="coerce") / 60acts_df["training_load"] = pd.to_numeric(acts_df.get("activityTrainingLoad", pd.Series(dtype=float)), errors="coerce")print(f"Total activities: {len(acts_df)}")print(f"Activity types:")print(acts_df["activityType.typeKey"].value_counts().to_string())print(f"Summary:")summary_cols = ["distance_km", "duration_min", "training_load", "averageHR"]for c in summary_cols:    if c in acts_df.columns:        s = pd.to_numeric(acts_df[c], errors="coerce").dropna()        if len(s):            print(f"  {c}: mean={s.mean():.1f}, min={s.min():.1f}, max={s.max():.1f}")

In [ ]:
# Training load per activityif "training_load" in acts_df.columns:    fig, axes = plt.subplots(1, 2, figsize=(14, 5))    ax = axes[0]    acts_df["date_dt"] = pd.to_datetime(acts_df["date"])    ax.bar(acts_df["date_dt"], acts_df["training_load"], color="steelblue", alpha=0.7)    ax.set_title("Training Load per Activity")    ax.set_ylabel("Training Load")    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))    ax = axes[1]    ax.scatter(acts_df["distance_km"], acts_df["duration_min"],               c=pd.to_numeric(acts_df.get("averageHR", pd.Series(dtype=float)), errors="coerce"),               cmap="YlOrRd", s=80, alpha=0.7, edgecolors="gray")    ax.set_xlabel("Distance (km)")    ax.set_ylabel("Duration (min)")    ax.set_title("Activities: Distance vs Duration (color = avg HR)")    plt.colorbar(ax.collections[0], ax=ax, label="Avg HR")    plt.tight_layout()    plt.show()

## 8. Activity Impact on RecoveryDoes training load affect next-day metrics?

In [ ]:
# Merge activity load with next-day recovery metricsif "act_total_training_load" in feats.columns:    impact = feats[["date", "act_total_training_load", "act_total_distance_km"]].copy()    # Next-day metrics    for c in ["sleep_score", "resting_hr", "hrv_last_night", "training_readiness_score"]:        if c in feats.columns:            impact[f"next_{c}"] = feats[c].shift(-1)    # Only days with actual training    trained = impact[impact["act_total_training_load"] > 0].copy()    rest = impact[impact["act_total_training_load"] == 0].copy()    print("Training days vs rest days — next-day metrics:")    for c in ["next_sleep_score", "next_resting_hr", "next_hrv_last_night", "next_training_readiness_score"]:        if c in impact.columns:            t_mean = trained[c].mean()            r_mean = rest[c].mean()            if pd.notna(t_mean) and pd.notna(r_mean):                print(f"  {c:40s} Train-day: {t_mean:.1f}  Rest-day: {r_mean:.1f}  Diff: {t_mean - r_mean:+.1f}")    if len(trained) > 2:        fig, axes = plt.subplots(1, 2, figsize=(14, 5))        for i, (metric, label) in enumerate([            ("next_hrv_last_night", "Next-day HRV"),            ("next_training_readiness_score", "Next-day Readiness"),        ]):            if metric in trained.columns:                ax = axes[i]                ax.scatter(trained["act_total_training_load"], trained[metric],                           s=80, alpha=0.7, edgecolors="gray")                ax.set_xlabel("Training Load")                ax.set_ylabel(label)                ax.set_title(f"Training Load vs {label}")                # Simple linear fit                mask = trained[["act_total_training_load", metric]].notna().all(axis=1)                if mask.sum() > 2:                    x = trained.loc[mask, "act_total_training_load"]                    y = trained.loc[mask, metric]                    z = np.polyfit(x, y, 1)                    p = np.poly1d(z)                    ax.plot(sorted(x), p(sorted(x)), "r--", alpha=0.5, label=f"slope={z[0]:.2f}")                    ax.legend()        plt.tight_layout()        plt.show()

## 9. Day-of-Week Patterns

In [ ]:
if "dow" in feats.columns:    dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]    dow_metrics = ["sleep_total_hrs", "sleep_score", "resting_hr", "hrv_last_night", "stress_avg"]    dow_avail = [c for c in dow_metrics if c in feats.columns]    fig, axes = plt.subplots(1, len(dow_avail), figsize=(4 * len(dow_avail), 5))    if len(dow_avail) == 1:        axes = [axes]    for ax, col in zip(axes, dow_avail):        grouped = feats.groupby("dow")[col].agg(["mean", "std", "count"])        ax.bar(range(7), grouped["mean"], yerr=grouped["std"], capsize=3,               color="steelblue", alpha=0.7)        ax.set_xticks(range(7))        ax.set_xticklabels(dow_labels, rotation=45)        ax.set_title(col)        # Show n per bar        for i, n in enumerate(grouped["count"]):            ax.text(i, grouped["mean"].iloc[i], f"n={int(n)}", ha="center", va="bottom", fontsize=8)    plt.suptitle("Day-of-Week Averages", y=1.02)    plt.tight_layout()    plt.show()

## 10. Key Scatter Plots

In [ ]:
scatter_pairs = [    ("sleep_total_hrs", "sleep_score", "Sleep Duration vs Score"),    ("hrv_last_night", "training_readiness_score", "HRV vs Training Readiness"),    ("resting_hr", "hrv_last_night", "Resting HR vs HRV"),    ("stress_avg", "sleep_score", "Avg Stress vs Sleep Score"),    ("bed_time_hour", "sleep_score", "Bedtime vs Sleep Score"),    ("body_battery_charged", "training_readiness_score", "BB Charged vs Readiness"),]valid_pairs = [(x, y, t) for x, y, t in scatter_pairs if x in clean.columns and y in clean.columns]n = len(valid_pairs)cols = 3rows = (n + cols - 1) // colsfig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))axes = axes.flatten() if n > 1 else [axes]for i, (xc, yc, title) in enumerate(valid_pairs):    ax = axes[i]    mask = clean[[xc, yc]].notna().all(axis=1)    ax.scatter(clean.loc[mask, xc], clean.loc[mask, yc], s=60, alpha=0.7, edgecolors="gray")    ax.set_xlabel(xc)    ax.set_ylabel(yc)    ax.set_title(title)    # Linear fit    if mask.sum() > 3:        x, y = clean.loc[mask, xc].values, clean.loc[mask, yc].values        z = np.polyfit(x, y, 1)        p = np.poly1d(z)        r = np.corrcoef(x, y)[0, 1]        xs = np.linspace(x.min(), x.max(), 50)        ax.plot(xs, p(xs), "r--", alpha=0.6)        ax.text(0.05, 0.95, f"r={r:.2f}", transform=ax.transAxes, va="top", fontsize=10,                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))for j in range(i + 1, len(axes)):    axes[j].set_visible(False)plt.tight_layout()plt.show()

## 11. Summary & Next Steps**Key observations** (fill in after running):- With only ~16 days, all correlations should be treated as preliminary- Watch for: RHR ↔ HRV inverse relationship, sleep duration ↔ readiness, stress ↔ body battery- Activity integration enables training load ↔ recovery analysis**Immediate next steps:**1. Continue collecting data — need 60+ days for meaningful regression2. Add manual labels (alcohol, caffeine, late meals, etc.) via a simple CSV3. Once ≥30 days: run OLS regression of sleep score on training load + stress + bed time4. Once ≥60 days: train a classifier for readiness flag (red/amber/green)5. Consider pulling in external data: weather (temp/humidity), calendar (busy days)

In [ ]:
print("=== Dataset Status ===")print(f"Days of data: {len(clean)}")print(f"Feature columns: {len(feats.columns)}")print(f"Activities logged: {len(acts)}")print(f"Target: Accumulate >= 60 days before serious modeling.")print(f"Days until 60-day threshold: {max(0, 60 - len(clean))}")